# LAB-HW-09 — DDR Integrity

**One new thing today:** prove a large deterministic payload survives a real KV260/K26 system-memory write/read roundtrip before you benchmark AXI or bursts.

Course sequence: after LAB-HW-08. Operational prerequisites: LSN-016, LSN-017, and a working LAB-HW-05 PS/Linux boot path.

**Project Trace:** RMD-014 · T-HW-009/T-HW-011

This Lab needs **no new bitstream**. That is intentional.

## 1. BRAM and DDR solve different problems

LAB-HW-07 used on-chip Block RAM (BRAM): small and close to programmable logic.

The KV260's K26 system has **4 GB DDR4** system memory. DDR provides much more capacity, but it sits behind the processing-system memory path rather than inside the PL fabric.

This Lab asks only: **can we store and recover known bytes reliably?**

It does not yet ask: “how fast can a PL AXI master move them?”

## 2. Freeze the layer we are testing

<svg xmlns="http://www.w3.org/2000/svg" width="980" height="310" viewBox="0 0 980 310" role="img" aria-label="LAB-HW-09 PS Linux managed DDR integrity path">
  <rect x="30" y="95" width="170" height="90" rx="10" fill="#eef0ff" stroke="#333"/>
  <text x="115" y="128" text-anchor="middle" font-size="14">Python helper</text>
  <text x="115" y="153" text-anchor="middle" font-size="12">PS / Linux</text>
  <rect x="250" y="95" width="180" height="90" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="340" y="128" text-anchor="middle" font-size="14">OS-managed</text>
  <text x="340" y="153" text-anchor="middle" font-size="12">anonymous mapping</text>
  <rect x="480" y="95" width="190" height="90" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="575" y="128" text-anchor="middle" font-size="14">PS memory path</text>
  <text x="575" y="153" text-anchor="middle" font-size="12">platform controller</text>
  <rect x="720" y="95" width="220" height="90" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="830" y="128" text-anchor="middle" font-size="14">K26 external DDR4</text>
  <text x="830" y="153" text-anchor="middle" font-size="12">4 GB system memory</text>
  <path d="M200 140 L250 140 M430 140 L480 140 M670 140 L720 140" stroke="#333" stroke-width="2"/>
  <polygon points="250,140 240,135 240,145" fill="#333"/><polygon points="480,140 470,135 470,145" fill="#333"/><polygon points="720,140 710,135 710,145" fill="#333"/>
  <rect x="480" y="225" width="220" height="55" rx="8" fill="#f4f4f4" stroke="#999" stroke-dasharray="7 5"/>
  <text x="590" y="258" text-anchor="middle" font-size="13">PL → AXI HP → DDR: LAB-HW-10</text>
</svg>

The dashed path is deliberately deferred.

We do **not** use raw `/dev/mem` for DDR here. Linux owns the allocation, so the Lab cannot accidentally overwrite the kernel, filesystem cache, or another process.

## 3. Frozen integrity contract

Physical mode always uses:

| item | value |
|---|---:|
| total test allocation | **64 MiB** |
| chunk size | **1 MiB** |
| chunks | 64 |
| access pattern | contiguous sequential chunks |
| payload | deterministic SHAKE256 chunk-index pattern |
| comparison | byte-for-byte + SHA-256 |
| root required | no |
| bitstream required | no |

Before timed writes, the helper **prefaults** the mapping so first-touch page allocation is not silently mixed into the payload-copy timer.

The payload generator is deterministic: rerunning the same helper regenerates the same bytes for the same chunk index.

## 4. Integrity comes before every bandwidth number

The acceptance order is strict:

```console
write deterministic bytes
→ read every chunk
→ byte-for-byte compare
→ compare expected/observed SHA-256
→ INTEGRITY=PASS
→ only then print timing/bandwidth observations
```

If one byte differs:

```console
INTEGRITY=FAIL
PERFORMANCE_BLOCKED=1
ERROR=DDR_INTEGRITY_MISMATCH
STATUS=FAIL
```

A broken data path never gets “rescued” by a fast timing number.

## 5. Dry-run the oracle before the board

On any development machine:

```bash
python boards/kv260/runtime/ddr_integrity.py \
  --dry-run \
  --json-out /tmp/lab-hw-09-dry-run.json
```

Dry-run uses 4 MiB so CI remains fast. It exercises the same deterministic generation, prefault, byte comparison, SHA-256 comparison, and “integrity before timing” logic.

Expected final markers:

```console
INTEGRITY=PASS
PERFORMANCE_BLOCKED=0
BANDWIDTH_SCOPE=HOST_PATH_OBSERVATION_NOT_PEAK_DDR_OR_PL_AXI
STATUS=PASS
```

This is checker evidence, not physical T-HW-009 evidence.

## 6. Prove that corruption really blocks performance

CI/test-only command:

```bash
python boards/kv260/runtime/ddr_integrity.py \
  --dry-run \
  --inject-corruption-for-test
```

The helper flips one byte after the write. The correct result is a non-zero exit with `DDR_INTEGRITY_MISMATCH` and `PERFORMANCE_BLOCKED=1`.

The corruption option is rejected in physical mode. We do not deliberately corrupt a student's real physical run.

## 7. Run the real KV260 memory sanity check

On the KV260 PS/Linux runtime host:

```bash
python3 /tmp/ddr_integrity.py \
  --physical \
  --json-out /tmp/lab-hw-09-trace.json
```

No `sudo` is required.

Physical mode verifies that `/proc/device-tree/model` identifies a Kria/KV260 environment, then records:

- board model;
- kernel and machine;
- OS identity;
- `MemTotal` and `MemAvailable`;
- swap totals;
- helper SHA-256;
- test geometry and access pattern.

If available memory is too low for a safe 64 MiB test, the Lab fails rather than forcing allocation under memory pressure.

## 8. What do the timing numbers mean?

After integrity PASS, the helper reports:

- `WRITE_ELAPSED_NS`
- `READ_ELAPSED_NS`
- `HOST_PATH_WRITE_MIB_PER_S`
- `HOST_PATH_READ_MIB_PER_S`

The timer boundaries are printed in the log.

These are **observations of this PS/Linux/userspace path**. They are not:

- peak DDR4 bandwidth;
- PL→DDR bandwidth;
- AXI burst efficiency;
- hardware-only latency;
- a CPU-vs-FPGA benchmark.

Caches and the operating system are part of this path. LAB-HW-10 introduces a controlled PL/AXI measurement contract.

## 9. Failure classes

The helper separates:

- `NOT_KV260_RUNTIME_ENVIRONMENT`
- `INSUFFICIENT_AVAILABLE_MEMORY`
- `MEMORY_TEST_SETUP_FAILED`
- `DDR_INTEGRITY_MISMATCH`
- `INJECTION_NOT_ALLOWED_PHYSICAL`

A runtime-environment failure is not a DDR data mismatch. An integrity mismatch is not a performance result.

If integrity fails, debug correctness first; do not start tuning access patterns.

## 10. What this Lab does not prove

A userspace write/read path can involve CPU caches. Therefore LAB-HW-09 does **not** claim that each timed load/store directly became one electrical DDR transaction.

That lower-level claim would require a different instrumentation/control boundary.

What HW-09 proves is narrower and useful: on the real KV260 PS/Linux system-memory path, a large deterministic working set can be allocated, written, recovered, and verified without corruption.

That is enough to earn the right to proceed to HW-10.

## 11. Expected Evidence / Save Evidence

Retain:

- `ddr_integrity.py` SHA-256;
- `lab-hw-09-trace.json`;
- complete terminal stdout;
- board model;
- kernel/OS identity;
- `MemTotal`, `MemAvailable`, swap snapshot;
- payload ID;
- 64 MiB total size and 1 MiB chunks;
- contiguous access-pattern label;
- expected and observed SHA-256;
- byte-compare PASS;
- timer-boundary strings;
- raw write/read elapsed nanoseconds;
- host-path MiB/s observations;
- Git commit and experiment date.

**No bitstream hash is required for LAB-HW-09**, because this Lab deliberately introduces no new PL design.

Ordinary CI dry-run remains non-physical evidence.

## 12. Troubleshooting order

1. Confirm you are on the KV260 PS/Linux runtime host.
2. Check `MemAvailable`; close large applications if necessary.
3. Re-run the dry-run helper to confirm the oracle itself still passes.
4. Re-run physical mode and compare expected/observed SHA-256.
5. If a mismatch repeats, retain the JSON and terminal evidence before changing anything.
6. Do not jump to AXI/burst tuning until integrity is stable.

## 13. Human Check

Explain:

1. Why does HW-09 use Linux-managed memory instead of a guessed `/dev/mem` DDR address?
2. Why is 64 MiB much larger than the HW-07 4 KiB BRAM teaching window?
3. Why do we use both byte-for-byte comparison and SHA-256?
4. Why does one corrupted byte block every performance conclusion?
5. Why are `HOST_PATH_*_MIB_PER_S` not peak DDR bandwidth?
6. Why is no new bitstream required?
7. What new concept is intentionally deferred to LAB-HW-10?

## 14. Engineering handoff

HW-09 gives us a correctness baseline for external system memory without introducing a new PL protocol problem.

Before HW-10, **stop here and complete the independent Ubuntu / u-dma-buf prerequisite**:

`docs/en/KV260_UDMABUF_SETUP.md`

On the KV260 runtime host, require:

```bash
sudo python3 /tmp/preflight_udmabuf.py \
  --physical \
  --json-out /tmp/lab-hw-10-udmabuf-preflight.json
```

Only after the prerequisite ends in `STATUS=PASS` continue to:

```console
LAB-HW-09
OS-managed DDR integrity
        ↓
Ubuntu/u-dma-buf setup + preflight
        ↓
LAB-HW-10
PL → AXI high-performance port → DDR
small/scattered vs contiguous/burst measurement
```

HW-10 may now change the access path while keeping the rule learned here: **corrupt data invalidates performance claims.**


## 15. Official basis

- AMD Kria KV260 Vision AI Starter Kit Data Sheet (DS986), Product Details: KV260 provides 4 GB non-ECC DDR4 system memory.
- AMD Zynq UltraScale+ MPSoC Processing System Product Guide (PG201), Slave Interface: PL high-performance `S_AXI_HP0..3_FPD` interfaces provide non-coherent PL paths toward the FPD main switch and DDR. That PL path is intentionally deferred to LAB-HW-10.
- Repository teaching source: `lessons/en/17_external_memory_ddr.ipynb` states that DDR hello-world proves integrity before bandwidth benchmarking.

Official AMD references:
- https://docs.amd.com/r/en-US/ds986-kv260-starter-kit/Product-Details
- https://docs.amd.com/r/en-US/pg201-zynq-ultrascale-plus-processing-system/Slave-Interface